In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [ ]:
# BB84 Quantum Key Distribution : simulation WITH an attacker (Eve)
#
# Eve intercepts every qubit, measures it in a random basis, then
# re-prepares and forwards a new qubit.  Because she guesses the
# basis correctly only ~50% of the time, she disturbs ~25% of the
# sifted bits : enough for Alice and Bob to detect her.

In [ ]:
# Quantum Random Number Generator
# Each qubit is prepared in (|0> + |1>) / sqrt(2) by applying H to |0>.
# Measuring this state yields 0 or 1 with equal probability — true quantum randomness.

def quantum_random_bits(n):
    """Return n random bits from quantum measurement of n H|0> qubits."""
    qc = QuantumCircuit(n, n)
    for i in range(n):
        qc.h(i)              # H|0> = (|0> + |1>) / sqrt(2)
    qc.measure(range(n), range(n))

    simulator = BasicSimulator()
    compiled  = transpile(qc, simulator)
    result    = simulator.run(compiled, shots=1).result()
    bitstring = list(result.get_counts().keys())[0]
    # Qiskit is little-endian: rightmost char = qubit 0, so reverse.
    return [int(b) for b in reversed(bitstring)]

In [ ]:
# ALICE
# (Identical to the plain protocol, Alice is unaware of Eve.)

N = 20   # number of qubits transmitted

alice_bits  = quantum_random_bits(N)
alice_bases = quantum_random_bits(N)  # 0 = rectilinear, 1 = diagonal

print("=== ALICE ===")
print(f"Bits:  {alice_bits}")
print(f"Bases: {alice_bases}  (0=+, 1=x)")

=== ALICE ===
Bits:  [1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0]
Bases: [0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0]  (0=+, 1=x)


In [ ]:
# EVE  (attacker : intercepts the quantum channel)

# Eve picks random measurement bases, measures each qubit
# (collapsing its state), then re-prepares and forwards a new
# qubit in the state she observed.
#
# When Eve's basis matches Alice's: her copy is correct, no disturbance.
# When they differ: Eve sends the wrong state 50% of the time,
# introducing a detectable error at Bob's end.
#
# Circuit A: Alice prepares, Eve measures (in her basis)
# Eve then re-encodes based on her result and sends to Bob.

eve_bases = quantum_random_bits(N)

print("=== EVE (secret) ===")
print(f"Bases: {eve_bases}  (0=+, 1=x)")
print()

# --- Circuit A: Alice -> Eve ---
qc_eve = QuantumCircuit(N, N)

for i in range(N):
    # Alice's encoding
    if alice_bits[i] == 1:
        qc_eve.x(i)
    if alice_bases[i] == 1:
        qc_eve.h(i)
    # Eve rotates into her measurement basis before measuring
    if eve_bases[i] == 1:
        qc_eve.h(i)

qc_eve.measure(range(N), range(N))

simulator = BasicSimulator()
compiled  = transpile(qc_eve, simulator)
result    = simulator.run(compiled, shots=1).result()
bitstring = list(result.get_counts().keys())[0]
eve_results = [int(b) for b in reversed(bitstring)]

print(f"Eve's measured bits (secret): {eve_results}")
print()
# Show how many bases Eve guessed correctly
eve_correct = sum(e == a for e, a in zip(eve_bases, alice_bases))
print(f"Eve's basis matched Alice's : {eve_correct}/{N} ({eve_correct/N:.0%})")
print("(Eve cannot know this)")

=== EVE (secret) ===
Bases: [0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0]  (0=+, 1=x)

Eve's measured bits (secret): [1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0]

Eve's basis matched Alice's : 14/20 (70%)
(Eve cannot know this)


In [ ]:
# QUANTUM CHANNEL (Eve -> Bob)  to  BOB

# Eve re-prepares each qubit in *her* measured basis and sends to Bob.
# Bob measures in his own randomly chosen basis.
#
# Circuit B: Eve re-encodes (Bob decodes) measure

bob_bases = quantum_random_bits(N)

print("=== BOB ===")
print(f"Bases: {bob_bases}  (0=+, 1=x)")
print()

# --- Circuit B: Eve -> Bob ---
qc_bob = QuantumCircuit(N, N)

for i in range(N):
    # Eve re-encodes in her basis using her measurement result
    if eve_results[i] == 1:
        qc_bob.x(i)
    if eve_bases[i] == 1:
        qc_bob.h(i)
    # Bob decodes in his chosen basis
    if bob_bases[i] == 1:
        qc_bob.h(i)

qc_bob.measure(range(N), range(N))

simulator = BasicSimulator()
compiled  = transpile(qc_bob, simulator)
result    = simulator.run(compiled, shots=1).result()
bitstring = list(result.get_counts().keys())[0]
bob_results = [int(b) for b in reversed(bitstring)]

print(f"Bob's measured bits: {bob_results}")

=== BOB ===
Bases: [1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1]  (0=+, 1=x)

Bob's measured bits: [0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0]


In [ ]:
# CLASSICAL CHANNEL : Basis Comparison  (public)

matching = [i for i in range(N) if alice_bases[i] == bob_bases[i]]

alice_sifted = [alice_bits[i]  for i in matching]
bob_sifted   = [bob_results[i] for i in matching]

print("=== BASIS COMPARISON ===")
header = f"{'Qubit':>5} | {'Alice':>6} | {'Bob':>6} | {'Keep?':>6}"
print(header)
print("-" * len(header))
for i in range(N):
    keep = "YES" if alice_bases[i] == bob_bases[i] else ""
    print(f"{i:>5} | {alice_bases[i]:>6} | {bob_bases[i]:>6} | {keep:>6}")

print(f"\nMatched {len(matching)} / {N} positions: {matching}")
print(f"\nAlice sifted key : {alice_sifted}")
print(f"Bob   sifted key : {bob_sifted}")

=== BASIS COMPARISON ===
Qubit |  Alice |    Bob |  Keep?
--------------------------------
    0 |      0 |      1 |       
    1 |      0 |      1 |       
    2 |      0 |      1 |       
    3 |      0 |      1 |       
    4 |      1 |      1 |    YES
    5 |      0 |      1 |       
    6 |      1 |      1 |    YES
    7 |      0 |      1 |       
    8 |      0 |      0 |    YES
    9 |      1 |      1 |    YES
   10 |      0 |      1 |       
   11 |      0 |      0 |    YES
   12 |      0 |      1 |       
   13 |      0 |      0 |    YES
   14 |      1 |      1 |    YES
   15 |      0 |      0 |    YES
   16 |      0 |      1 |       
   17 |      1 |      0 |       
   18 |      0 |      1 |       
   19 |      0 |      1 |       

Matched 8 / 20 positions: [4, 6, 8, 9, 11, 13, 14, 15]

Alice sifted key : [0, 0, 1, 1, 1, 0, 0, 0]
Bob   sifted key : [0, 1, 1, 1, 0, 0, 0, 0]


In [ ]:
# ATTACK DETECTION

# Alice and Bob sacrifice a portion of their sifted key bits by
# comparing them publicly.  A non-zero error rate exposes Eve.
#
# Expected error rate WITH Eve intercepting all qubits: ~25%
#   (Eve wrong basis 50% of time, causes 50% error when wrong -> 25% overall)
# Expected error rate WITHOUT attacker: 0%
# Detection threshold: 11% (well below expected Eve noise of 25%)

THRESHOLD = 0.11

errors     = sum(a != b for a, b in zip(alice_sifted, bob_sifted))
error_rate = errors / len(alice_sifted) if alice_sifted else 0.0

print("=== ATTACK DETECTION ===")
print(f"Sifted key length : {len(alice_sifted)} bits (all compared publicly)")
print(f"Errors found      : {errors}")
print(f"Error rate        : {error_rate:.1%}")
print(f"Detection threshold: {THRESHOLD:.0%}")
print()
if error_rate > THRESHOLD:
    print("[ATTACK DETECTED] Error rate exceeds threshold.")
    print("Alice and Bob abort the protocol — the channel is compromised.")
else:
    print("[No attack detected] Error rate within acceptable bounds.")
    print("(With only 20 qubits Eve may get lucky — increase N for reliability)")
print()

# Summary for educational clarity
print("--- Summary ---")
print(f"Alice's bases : {alice_bases}")
print(f"Eve's bases   : {eve_bases}")
print(f"Bob's bases   : {bob_bases}")
print()
print(f"Alice bits    : {alice_bits}")
print(f"Eve results   : {eve_results}  (secret)")
print(f"Bob results   : {bob_results}")

=== ATTACK DETECTION ===
Sifted key length : 8 bits (all compared publicly)
Errors found      : 2
Error rate        : 25.0%
Detection threshold: 11%

[ATTACK DETECTED] Error rate exceeds threshold.
Alice and Bob abort the protocol — the channel is compromised.

--- Summary ---
Alice's bases : [0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0]
Eve's bases   : [0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0]
Bob's bases   : [1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1]

Alice bits    : [1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0]
Eve results   : [1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0]  (secret)
Bob results   : [0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0]
